# **자료구조 기초**



---



## **1. 자료 구조의 개요**

## **2. 자료구조와 알고리즘**


### **알고리즘 관점에서 데이터 구조가 왜 중요할까?**

### **예시1:** 탐색 자료구조별 성능 비교
```
❓100만개의 데이터 중 특정 한 개를 찾는 경우에 다음 중 어떤 자료 구조가 더 효율적일까?
```
| 자료구조 | 시간 복잡도 | 100만 개 데이터 시뮬레이션 결과 | 검색 사례 적용 |
| :--- | :---: | :--- | :--- |
| **Array (배열)** | $O(n)$ | **1,000,000번** 일일이 비교 | 모든 웹페이지를 첫 페이지부터 끝까지 뒤지는 방식 (**불가능**) |
| **BST (균형 탐색 트리)** | $O(\log n)$ | 단 **20번**의 비교로 끝남: log₂(1,000,000) | 데이터를 계층적으로 분류하여 탐색 범위를 순식간에 좁힘 |
| **Hash Table (해시)** | $O(1)$ | 평균 **1번**에 즉시 탐색 | 검색어를 입력하면 해당 데이터의 주소로 바로 점프함 (**최적**) |


In [ ]:
import time
import random

# 데이터 설정 (100만 개)
n = 1000000
target = random.randint(0, n-1)
data_list = list(range(n))          # 배열 (선형 탐색용, 이진 탐색용으로도 사용 가능)
random.shuffle(data_list)           # 데이터 섞기
data_set = set(data_list)           # 해시 테이블 (Python의 set/dict)


# 1. 선형 탐색 (Array) - O(n)
def linear_search(data, target):
    count = 0
    for item in data:
        count += 1
        if item == target:
            return count
    return count


# 2. 이진 탐색 (실제 구현) - O(log n)
# 정렬된 배열에서 target을 찾고 연산 횟수를 반환
def binary_search_count(data, target):
    low = 0
    high = len(data) - 1
    count = 0

    while low <= high:
        count += 1
        mid = (low + high) // 2

        if data[mid] == target:
            return count # 타겟을 찾으면 연산 횟수 반환
        elif data[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return count # 타겟을 찾지 못했어도 연산 횟수는 반환 (최악의 경우)


# --- 성능 측정 ---

# [Array]
start_time = time.time()
linear_ops = linear_search(data_list, target)
linear_time = time.time() - start_time

# [BST / 이진 탐색]
# 실제 이진 탐색 함수를 호출하여 연산 횟수 측정
start_time_bst = time.time() # BST 시간 측정 시작
bst_ops = binary_search_count(data_list, target)
bst_time = time.time() - start_time_bst # BST 시간 측정 종료

# [Hash Table]
start_time = time.time()
found = target in data_set # Python set은 Hash Table 기반 O(1)
hash_time = time.time() - start_time
hash_ops = 1 # 평균적으로 단 1번의 해시 연산으로 접근

print(f"# 목표 데이터: {target} 찾기")
print("-" * 40)
print(f" - Array (선형 탐색):  약 {linear_ops:,}번 연산 | 소요 시간: {linear_time:.6f}초")
print(f" - BST (이진 탐색):    약 {bst_ops}번 연산     | 소요 시간: {bst_time:.6f}초") # 이론적 예측치 제거
print(f" - Hash Table (해시):  약 {hash_ops}번 연산      | 소요 시간: {hash_time:.6f}초")
print("-" * 40)
print(f"# 결과: 해시 테이블이 선형 탐색보다 약 {linear_ops / hash_ops:,.0f}배 적은 연산으로 데이터를 찾았습니다!")


- **결론:**
    1. 선형 탐색 (Array): 모든 요소를 하나씩 확인하므로 데이터 양에 비례해서 시간이 늘어남.
    2. 이진 탐색 (BST): 매 시도마다 확인할 데이터 양을 50%씩 제거함. (2의 20승 ≒ 100만)
    3. 해시 탐색 (Hash Map): 산술 연산을 통해 데이터의 위치(주소)를 즉시 계산함.

### **예시2:** List와 Heap 자료구조 성능 비교
```
❓네이버 뉴스 댓글에서 '좋아요' 순으로 실시간 정렬을 유지하려고 할 때, 단순 리스트 정렬과 힙(Heap) 중 어떤 것이 서버 부하가 적을까?
```
* **List**: 데이터를 순차적으로 저장하며 인덱스로 임의 접근(Random Access)이 가능한 구조
* **Heap**: 최댓값/최솟값 추적에 특화된 부모-자식 간의 크기 관계가 일정한 이진 트리 구조, Top-k 문제에 강함

In [ ]:
import heapq
import random
import time

# 1. 단순 리스트 정렬 방식 (매번 전체 정렬)
def list_sorting_approach(data, new_val):
    data.append(new_val)
    data.sort(reverse=True) # 매번 전체 데이터를 다시 정렬함 (비효율적)
    return data[0] # 가장 높은 '좋아요' 반환


# 2. 힙(Heap) 방식 (우선순위 큐 활용)
def heap_approach(heap_data, new_val):
    # 파이썬 heapq는 최소 힙이므로, 최대 힙처럼 쓰기 위해 음수로 저장
    heapq.heappush(heap_data, -new_val) # 로그 시간(log n) 안에 자기 자리 찾아감
    return -heap_data[0] # 가장 높은 '좋아요' 반환


# --- 성능 테스트 ---
n = 10000 # 현재 댓글 수 1만 개
initial_likes = [random.randint(1, 1000) for _ in range(n)]

# 리스트 방식 측정
list_data = initial_likes[:]
start = time.time()
for _ in range(100): # 새로운 '좋아요'가 100번 발생했다고 가정
    list_sorting_approach(list_data, random.randint(1, 1000))
list_time = time.time() - start
print(f" - 단순 리스트 정렬 방식 소요 시간: {list_time:.4f}초")


# 힙 방식 측정
heap_data = [-x for x in initial_likes]
heapq.heapify(heap_data)
start = time.time()
for _ in range(100):
    heap_approach(heap_data, random.randint(1, 1000))
heap_time = time.time() - start
print(f" - 힙(Heap) 방식 소요 시간: {heap_time:.4f}초")

# 기존 측정 코드 아래에 추가
print('-'*50)
print(f"# 결과: 힙 방식이 리스트 정렬 방식보다 약 { (list_time / heap_time) :.4f}배 더 빠릅니다!")



---



## **3. 기초 자료구조**

### 3-1. **배열(Array)**
- 동일한 타입의 요소들이 연속적인 메모리 공간에 저장된 자료구조
- 인덱스(Index)를 사용하여 요소에 빠르게 접근
- 데이터 검색과 순차적인 데이터 처리를 위해 최적화된 자료구조

#### **배열의 특징**

|특징|설명|
|----|----|
|고정된 크기 (Static Size)  |배열은 일반적으로 선언 시 크기가 고정됨 (동적 배열은 예외)  |
|연속적인 메모리 할당 (Contiguous Memory Allocation)  |모든 요소가 연속된 메모리 주소에 저장됨 → 빠른 접근 가능 (𝑂(1))  |
|인덱스를 이용한 빠른 접근 (Index-Based Access)  |특정 요소에 𝑂(1) 의 시간 복잡도로 접근 가능.  |
|데이터 추가/삭제가 비효율적  |배열 크기가 고정되어 있으며, 요소 삽입/삭제 시 비효율적(𝑂(𝑛))  |

        

- **고정된 크기** (Static Size)

In [ ]:
// 고정된 크기 (Static Size) : C 언어
#include <stdio.h>
#define SIZE 5  // 배열 크기 정의

int main() {
    int fixed_array[SIZE] = {1, 2, 3, 4, 5}; // 크기가 5인 정수형 배열 선언

    // 배열 요소 출력
    printf("배열 요소: ");
    for (int i = 0; i < SIZE; i++) {
        printf("%d ", fixed_array[i]);
    }

    // 크기를 초과한 요소 추가 시 오류 (컴파일러에서 허용하지 않음)
    fixed_array[5] = 6;

    return 0;
}

In [ ]:
# 고정된 크기 (Static Size)
'''
파이썬의 기본 list는 동적 배열(dynamic array)이어서
고정된 크기처럼 사용하기 위해 임의의 class를 만들어 테스트
'''
import array

class FixedSizeArray:
    def __init__(self, size):
        self.array = array.array('i', [0] * size)  # 고정 크기 배열 초기화
        self.size = size
        self.count = 0  # 현재 저장된 요소 개수

    def insert(self, value):
        if self.count < self.size:
            self.array[self.count] = value
            self.count += 1
        else:
            raise ValueError("배열 크기를 초과하여 삽입할 수 없습니다!")

    def display(self):
        print(self.array[:self.count])  # 실제 저장된 요소만 출력

# 고정 크기 배열 생성 (크기 5)
fixed_array = FixedSizeArray(5)
fixed_array.insert(10)
fixed_array.insert(20)
fixed_array.insert(30)
fixed_array.insert(40)
fixed_array.insert(50)

fixed_array.display()  # 출력: array('i', [10, 20, 30, 40, 50])

# 크기 초과 시 오류 발생
try:
    fixed_array.insert(60)  # 오류 발생
except Exception as e:
    print("오류 발생:", e)


- **연속적인 메모리 할당** (Contiguous Memory Allocation)

In [ ]:
import array

arr = array.array('i', [10, 20, 30, 40, 50])

# 배열의 메모리 주소 확인
print(f"배열 시작 주소: {id(arr)}")
print(f"첫 번째 요소의 주소: {id(arr[0])}")
print(f"두 번째 요소의 주소: {id(arr[1])}")  # 연속적인 메모리 주소 확인


- **인덱스를 이용한 빠른 접근** (Index-Based Access)

In [ ]:
arr = [100, 200, 300, 400, 500]

# 인덱스를 이용한 O(1) 접근
print(arr[0])  # 출력: 100
print(arr[2])  # 출력: 300
print(arr[4])  # 출력: 500

- **데이터 추가/삭제가 비효율적** : 요소들이 이동해야 한다.

In [ ]:
arr = [10, 20, 30, 40, 50]

# 인덱스 2 위치에 25 삽입 (30 앞에 추가됨)
arr.insert(2, 25)
print(arr)  # 출력: [10, 20, 25, 30, 40, 50]

arr.pop(2)
print(arr)  # 출력: [10, 20, 40, 50]

#### **배열의 주요 연산** 및 시간 복잡도

|연산|설명|시간 복잡도|
|----|----|----|
|접근 (Access)|특정 인덱스의 요소를 가져오기  | O(1) |
|검색 (Search)  |특정 요소 찾기 (선형 검색)  | O(n) |
|삽입 (Insert)  |특정 위치에 요소 삽입  |O(n)  |
|삭제 (Delete)  |특정 위치 요소 삭제  |O(n)  |

In [ ]:
# 접근(Access) : O(1)
arr = [10, 20, 30, 40, 50]

# O(1) - 특정 인덱스의 요소 접근
print("인덱스 2의 값:", arr[2])  # 출력: 30
print("인덱스 4의 값:", arr[4])  # 출력: 50

In [ ]:
# 검색(Search) : O(n)
def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            return i  # 인덱스 반환
    return -1  # 요소가 없을 경우

arr = [10, 20, 30, 40, 50]
print("30의 위치:", linear_search(arr, 30))  # 출력: 2
print("100의 위치:", linear_search(arr, 100))  # 출력: -1 (없음)


In [ ]:
# 삽입(Insert) : O(n)
arr = [10, 20, 30, 40, 50]

# O(1) - 배열의 끝에 요소 추가
arr.append(60)
print("배열 끝에 60 추가:", arr)  # 출력: [10, 20, 30, 40, 50, 60]

# O(n) - 배열 중간에 요소 삽입 (인덱스 2에 25 삽입)
arr.insert(2, 25)
print("배열 중간에 25 삽입:", arr)  # 출력: [10, 20, 25, 30, 40, 50, 60]


In [ ]:
# 삭제(Delete) : O(n)
arr = [10, 20, 30, 40, 50]

# O(1) - 배열의 끝 요소 삭제
arr.pop()
print("배열 끝 요소 삭제:", arr)  # 출력: [10, 20, 30, 40]

# O(n) - 배열 중간 요소 삭제 (인덱스 2 요소 삭제)
arr.pop(2)
print("배열 중간 요소 삭제:", arr)  # 출력: [10, 20, 40]


#### **배열의 종류**

- **1차원  배열(One-Dimensional Array)** :
가장 기본적인 형태의 배열

In [ ]:
arr = [10, 20, 30, 40, 50]
print(arr[2])  # 출력: 30

- **2차원  배열 (Two-Dimensional Array)** :  행(row)과 열(column)로 구성된 배열 (행렬)


In [ ]:
matrix = [[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]

print(matrix[1][2])  # 출력: 6

- **다차원  배열 (Multi-Dimensional Array)** :  3차원 이상의 배열, 이미지 처리, 그래픽 연산, 머신러닝에서 활용  


In [ ]:
# 3D 배열 (2x2x2 배열)
array_3d = [
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
]

# 특정 요소 접근 (첫 번째 블록, 두 번째 행, 두 번째 열 -> 값: 4)
print("array_3d[0][1][1]:", array_3d[0][1][1])  # 출력: 4

# 3D 배열 출력
print("3차원 배열:")
for matrix in array_3d:
    for row in matrix:
        print(row)


In [ ]:
import numpy as np

# 2D 배열 (NumPy 사용)
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("NumPy 2D 배열:")
print(matrix)

# 특정 요소 접근 (행: 1, 열: 2 -> 값: 6)
print("matrix[1, 2]:", matrix[1, 2])  # 출력: 6


In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# 이미지 파일 읽기
image = Image.open("cat.jpg")

# NumPy 배열로 변환
image_array = np.array(image)

# 다차원 배열 확인
print("이미지 배열 형태:", image_array.shape)  # (높이, 너비, 채널)
print("배열 차원 수:", image_array.ndim)  # 3이면 컬러 이미지

# Matplotlib을 이용하여 이미지 출력
plt.imshow(image)
plt.axis("off")  # 축 제거
plt.show()


- **동적 배열 (Dynamic Array)** : 가장 크기가 가변적으로 조정됨(Python list, C++ vector) 요소 추가 시 자동으로 메모리 재할당 및 확장-

In [ ]:
arr = [1, 2, 3, 4, 5]
print("초기 배열:", arr)

# 요소 추가 (동적 크기 증가)
arr.append(6)
print("요소 추가 후:", arr)

# 요소 삭제 (동적 크기 감소)
arr.pop()
print("요소 삭제 후:", arr)


In [ ]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5])
print("초기 배열:", arr)

# 요소 추가 (새 배열 생성)
arr = np.append(arr, 6)
print("요소 추가 후:", arr)

# 요소 삭제 (새 배열 생성)
arr = np.delete(arr, -1)  # 마지막 요소 삭제
print("요소 삭제 후:", arr)


### **[실습]: 배열 구조 출력하기**
임의의 정수 12개를 발생시켜 1차원, 2차원(3x4), 3차원(2x2x3) 배열 구조로 출력하기

In [ ]:
import random

# 1~100 사이의 랜덤 정수 9개 생성
numbers = [random.randint(1, 100) for _ in range(12)]

# 1차원 배열 출력
print("1차원 배열:")
print(numbers)

# 2차원 배열 (3x4) 변환 및 출력
array_2d = [numbers[i:i+3] for i in range(0, 9, 3)]
print("\n2차원 배열 (3x4):")
for row in array_2d:
    print(row)

# 3차원 배열 (2x2x3) 변환 및 출력
array_3d = [[[numbers[k * 6 + i * 3 + j] for j in range(3)] for i in range(2)] for k in range(2)]

# 출력
print("3차원 배열 (2x2x3):")
for layer in array_3d:
    for row in layer:
        print(row)
    print()  # 층 구분

In [ ]:
import numpy as np

# 1~100 사이의 랜덤 정수 9개 생성
# numbers = np.random.randint(1, 101, 12)
numbers = np.array(numbers)

# 1차원 배열 출력
print("1차원 배열:")
print(numbers)

## 2차원 배열 (3x4) 변환 및 출력
array_2d = numbers.reshape(3, 4)
print("\n2차원 배열 (3x4):")
print(array_2d)

# 3차원 배열 (2x2x3) 변환 및 출력
array_3d = numbers.reshape(2, 2, 3)
print("\n3차원 배열 (2x2x3):")
print(array_3d)




---



### 3-2.**스택(Stack)**

- 다른 통로들은 모두 막고 한쪽만을 열어둔 구조
- 후입선출(LIFO:Last-In First-Out)/선입후출(FILO)의 자료구조

#### **스택의 연산**
|연산|설명|
|---|---|
|push(e) |스택의 맨 위에 데이터(요소) 삽입 |
|pop() |스택의 맨 위에 있는 요소 제거 |
|peek/top() |스택의 맨 위에 있는 항목을 삭제하지 않고 반환,  데이터 참조  |
|isEmpty() |스택이 비어 있는지 확인, True/False 반환 |
|isFull() |스택이 가득 차 있는지 확인, True/False 반환 |
|size() |스택에 들어 있는 전체 요소의 수 반환 |


#### **배열로 스택 구현하기**

- 전역변수 지정하기

In [ ]:
capacity = 10            # 스택 용량: 10으로 지정
array = [None]*capacity  # 요소 배열: [None,..,None] 길이(capacity)
top = -1                 # 스택 상단의 인덱스 : 공백 상태(-1)로 초기화

- 스택 연산 - 공백 상태 : isEmpty()

In [ ]:
def isEmpty():
    if top == -1:
        return True
    else:
        return False

- 스택 연산 - 포화 상태 : isEmpty()

In [ ]:
def isFull():
    return top == capacity    # 비교 연산 결과를 바로 반환

- 스택 연산 - 추가 : push(e)

In [ ]:
def push(e):
    global top
    if not isFull():             # 포화 상태가 아닌 경우
        top += 1                 # top 증가()
        array[top] = e           # top 위치에 e 복사
    else:                        # 포화 상태 : overflow
        print('stack overflow!')
        exit()

- 스택 연산 - 삭제 : pop()

In [ ]:
def pop():
    global top
    if not isEmpty():            # 공백 상태가 아닌 경우
        top -= 1                 # top 감소
        return array[top+1]      # 이전(top+1) 위치의 요소 반환
    else:                        # 공백 상태 : underflow
        print('stack underflow!')
        exit()

- 스택 연산 - 상단 요소 참조 : peek()

In [ ]:
def peek():
    if not isEmpty():
        return array[top]
    else:
        print("Stack is empty!")
        exit()

- 스택 연산 - 현재 스택 요소 수(크기) 반환 : size()

In [ ]:
def size():
    return top+1     # 현재 요소의 수

### **[실습]: 스택 클래스 구현하기**

In [ ]:
# 1. ArrayStack 이란 이름의 빈 스택 클래스 작성하기 (사이즈 제한 없는)
class ArrayStack:
    def __init__(self):
        """ 스택 초기화 """
        self.stack = []

In [ ]:
# 2. ArrayStack에 push(), pop(), peek(), isEmpty(), size() 메서드 추가하기
class ArrayStack:
    def __init__(self):
        """ 스택 초기화 """
        self.stack = []

    def push(self, item):
        """ 스택에 요소 추가 """
        self.stack.append(item)

    def pop(self):
        """ 스택에서 요소 제거 및 반환 """
        if self.isEmpty():
            print("Stack underflow!.")
            return None
        return self.stack.pop()

    def peek(self):
        """ 스택의 최상위(top) 요소 반환 (제거하지 않음) """
        if self.isEmpty():
            print("Stack overflow!.")
            return None
        return self.stack[-1]

    def isEmpty(self):
        """ 스택이 비어있는지 확인 """
        return len(self.stack) == 0

    def size(self):
        """ 현재 스택의 크기 반환 """
        return len(self.stack)

In [ ]:
# 3. 스택 연산 실행하기
# 스택 인스턴스 생성
stack = ArrayStack()

# 스택 연산
print(f'(a) {stack.stack}')
stack.push('A'); print(f'(b) {stack.stack}')
stack.push('B'); print(f'(c) {stack.stack}')
stack.push('C'); print(f'(d) {stack.stack}')
print(f'(e) {stack.pop()}')
print(f'(f) {stack.peek()}')
print(f'(g) {stack.isEmpty()}')

In [ ]:
# 4. Stack에 사이즈 제한 있도록 isFull() 매서드 추가하기
class ArrayStack:
    def __init__(self, capacity):
        """ 스택 초기화 (최대 크기 지정 가능) """
        self.stack = []
        self.capacity = capacity

    def push(self, item):
        """ 스택에 요소 추가 (최대 크기 제한) """
        if self.isFull():
            print("Stack is full! Cannot push.")
        else:
            self.stack.append(item)

    def pop(self):
        """ 스택에서 요소 제거 및 반환 """
        if self.isEmpty():
            print("Stack is empty! Cannot pop.")
            return None
        return self.stack.pop()

    def peek(self):
        """ 스택의 최상위(top) 요소 반환 (제거하지 않음) """
        if self.isEmpty():
            print("Stack is empty! Nothing to peek.")
            return None
        return self.stack[-1]

    def isEmpty(self):
        """ 스택이 비어있는지 확인 """
        return len(self.stack) == 0

    def size(self):
        """ 현재 스택의 크기 반환 """
        return len(self.stack)

    def isFull(self):
        """ 스택이 가득 찼는지 확인 """
        return len(self.stack) >= self.capacity


In [ ]:
# 사용 예시
stack = ArrayStack(5)  # 스택 크기 5

# 스택 연산
print(f'(a) {stack.stack}')
stack.push('A'); print(f'(b) {stack.stack}')
stack.push('B'); print(f'(c) {stack.stack}')
stack.push('C'); print(f'(d) {stack.stack}')
print(f'(e) {stack.pop()}')
print(f'(f) {stack.peek()}')
print(f'(g) {stack.isEmpty()}')

# 스택이 가득 찬 경우 테스트
stack.push('D')
stack.push('E')
stack.push('F')
stack.push('G')  # Stack is full! Cannot push.

# 최종 상태 출력
print("Final Stack size:", stack.size())
print("Is stack full?", stack.isFull())


#### **스택의 응용: 괄호 검사**

**괄호 검사 알고리즘 조건**
- 조건1 :  왼쪽 괄호의 개수와 오른쪽 괄호의 개수가 같아야 한다.
- 조건2 :  같은 종류인 경우 왼쪽 괄호가 오른쪽보다 먼저 나와야 한다.
- 조건3 :  다른 종류의 괄호 쌍이 서로 교차하면 안 된다.

### **[실습]: 괄호 검사 알고리즘 구현하기**

- 빈 스택을 준비합니다.
- 입력된 문자를 하나씩 읽어 왼쪽 괄호를 만나면 스택에 삽입합니다.
- 오른쪽 괄호를 만나면 가장 최근에 삽입된 괄호를 스택에서 꺼냅니다. 이때 스택이 비었다면 오
른쪽 괄호가 먼저 나온 상황이므로 조건 2에 위배됩니다.
- 꺼낸 괄호가 오른쪽 괄호와 짝이 맞지 않으면 조건 3에 위배됩니다.
- 입력 문자열을 끝까지 처리했는데 스택에 괄호가 남아 있으면 괄호의 개수가 같지 않으므로 조
건 1에 위배됩니다. 모든 문자를 처리하고 스택이 공백 상태이면 검사 성공입니다.

1.괄호 검사 알고리즘 구현하기

In [ ]:
def checkBrackets(statement):
    stack = ArrayStack(100)  # 빈 스택을 준비합니다.

    # 입력된 문자를 하나씩 읽어
    for ch in statement:
        # 왼쪽 괄호를 만나면 스택에 삽입합니다.
        if ch in ('[','{','('):    # 열린 괄호
            stack.push(ch)
        # 오른쪽 괄호를 만나면 가장 최근에 삽입된 괄호를 스택에서 꺼냅니다.
        # 이때 스택이 비었다면 오른쪽 괄호가 먼저 나온 상황이므로 조건 2에 위ㅎ배
        elif ch in (']','}',')'):  # 닫힌 괄호
            if stack.isEmpty():
                return False       # 조건2 위반
            else:
                left = stack.pop() # 문자를 pop해서 비교
                # 꺼낸 괄호가 오른쪽 괄호와 짝이 맞지 않으면 조건 3에 위배
                if  (ch == "]" and left != "[") or \
                    (ch == "}" and left != "{") or \
                    (ch == ")" and left != "(") :
                    return False    # 조건3 위반
    # 끝까지 처리했는데 스택에 괄호가 남아 있으면 괄호의 개수가 맞지 않아 조건 1에 위배됨.
    # 스택이 공백 상태면 검사 성공
    return stack.isEmpty()

2.예제를 이용하여 괄호 검사 알고리즘 검증하기

In [ ]:
str1 = "{ A[(i+1)]=0; }"
str2 = "if ((x<0) && (y<3)"
str3 = "while (n<8)) {n++;}"
str4 = "arr[(i+1])=0;"

print(str1, " ---> \t", checkBrackets(str1))
print(str2, " ---> ", checkBrackets(str2))
print(str3, " --->", checkBrackets(str3))
print(str4, " ---> \t", checkBrackets(str4))

#### **[참고] 파이썬에서 스택 사용하기**
- 예제: 문자열 역순으로 출력하기 사용

##### 방법 1) Stack 클래스 직접 구현해서 사용하기

In [ ]:
stack = ArrayStack(100) # 스택 생성

# 1.키보드로 문자열 입력 받기
msg = input('문자열 입력: ')
print('-' * 30)

# 2.문자열 스택에 추가하기
for c in msg:
    stack.push(c)

# 3.스택에서 문자열 꺼내서 출력하기
print('문자열 거꾸로 출력: ', end='')
while not stack.isEmpty():
    print(stack.pop(), end='')

##### 방법 2) 파이썬 리스트 함수 사용해서 스택으로 사용하기

In [ ]:
s = list()                      # 리스트를 객체를 생성해 스택으로 사용

msg = input("문자열 입력: ")
for c in msg :
    s.append(c)                 # c를 스택에 삽입

print("문자열 출력: ", end='')
while len(s) > 0:               # 스택이 공백상태가 아니라면
    print(s.pop(), end='')      # 하나의 요소를 꺼내서 출력
print()

##### 방법 3) 파이썬의 queue 모듈 LifoQueue 사용하기

In [ ]:
import queue                     # 파이썬의 큐 모듈 포함
s = queue.LifoQueue(maxsize=100) # 스택 객체 생성(용량=100)

msg = input("문자열 입력: ")
for c in msg :
    s.put(c)                     # c를 스택에 삽입

print("문자열 출력: ", end='')
while not s.empty():             # 스택이 공백상태가 아니라면
    print(s.get(), end='')       # 하나의 요소를 꺼내서 출력
print()



---



### 3-3. **큐(Queue)**

- 선입선출(FIFO:First-In First-Out)의 자료구조
- 가장 먼저 들어온 데이터가 가장 먼저 나감

#### **큐의 연산**
|연산|설명|
|---|---|
|enqueue(e) |새로운 요소 e를 큐의 맨 뒤에 추가 |
|dequeue() |큐의 맨 앞에 있는 요소를 꺼내서 반환 |
|peek |큐의 맨 위에 있는 요소를 삭제하지 않고 반환  |
|isEmpty() |큐가 비어 있는지 확인, True/False 반환 |
|isFull() |큐가 가득 차 있는지 확인, True/False 반환 |
|size() |큐에 들어 있는 전체 요소의 수 반환 |


### **[실습]: 배열로 원형 큐 클래스 구현하기**
```용량이 고정된 원형 큐 클래스```

- 1.큐 클래스(ArrayQueue) 생성하기

In [ ]:
# 1.(원형 큐) : 클래스 정의 와 생성자
class ArrayQueue:
    def __init__(self, capacity=5):
        self.capacity = capacity
        self.array = [None] * capacity
        self.front = 0
        self.rear = 0

    # 2.(원형 큐) : 공백 상태와 포화 상태 검사
    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return self.front == (self.rear+1)%self.capacity

    # 3.(원형 큐) : 삽입 연산
    def enqueue(self, item):
        if not self.isFull():
            self.rear = (self.rear+1)%self.capacity # 후단 회전
            self.array[self.rear] = item
        else:
            print('stack overflow!')
            pass

    # 4.(원형 큐) : 삭제 연산
    def dequeue(self):
        if not self.isEmpty():
            self.front = (self.front+1)%self.capacity  # 전단 회전
            return self.array[self.front]

    # 5. (원형 큐) : 상단 참조 연산
    def peek(self):
        if not self.isEmpty():
            return self.array[(self.front+1)%self.capacity]
        else: pass

    # 6. (원형 큐) : 전체 요소 수
    def size(self):
        return (self.rear - self.front + self.capacity) % self.capacity

    # 7. (원형 큐) : 전체 요소 화면에 출력
    def display(self, msg='Queue: '):
        print(msg, end='=[')
        count = self.size()
        for i in range(count):
            print(self.array[(self.front+1+i)%self.capacity], end=' ')
        print("]")

- 2.큐 테스트 프로그램 실행하기

In [ ]:
import random
q = ArrayQueue(8)                    # 큐 객체를 생성(capacity=8)

q.display("초기 상태")
while not q.isFull() :               # 큐에 빈 칸인 남았으면
    q.enqueue(random.randint(0,100)) # 0~99사이의 정수 발생->삽입
q.display("포화 상태")

print("삭제 순서: ", end='')
while not q.isEmpty() :             # 큐에 요소가 남아 있으면
    print(q.dequeue(), end=' ')     # 꺼내서 화면에 출력
print()

### **[실습] 원형 큐를 링 버퍼로 구현하기**

- 1.링 버퍼 적용한 클래스

In [ ]:
# 1.(원형 큐) : 클래스 정의 와 생성자
class ArrayQueue:
    def __init__(self, capacity=10):
        self.capacity = capacity
        self.array = [None] * capacity
        self.front = 0
        self.rear = 0

    # 2.(원형 큐) : 공백 상태와 포화 상태 검사
    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return self.front == (self.rear+1)%self.capacity

    # 3.(원형 큐) : 삽입 연산
    def enqueue(self, item):
        if not self.isFull():
            self.rear = (self.rear+1)%self.capacity # 후단 회전
            self.array[self.rear] = item
        else:
            print('stack overflow!')
            pass

    # 4.(원형 큐) : 삭제 연산
    def dequeue(self):
        if not self.isEmpty():
            self.front = (self.front+1)%self.capacity  # 전단 회전
            return self.array[self.front]

    # 5. (원형 큐) : 상단 참조 연산
    def peek(self):
        if not self.isEmpty():
            return self.array[(self.front+1)%self.capacity]
        else: pass

    # 6. (원형 큐) : 전체 요소 수
    def size(self):
        return (self.rear - self.front + self.capacity) % self.capacity

    # 7. (원형 큐) : 전체 요소 화면에 출력
    def display(self, msg='Queue: '):
        print(msg, end=' = [')
        count = self.size()
        for i in range(count):
            print(self.array[(self.front+1+i)%self.capacity], end=' ')
        print("]")

    # 8. (원형 큐) : 링 버퍼를 위한 삽입 연산
    def enqueue2(self, item):             # 링 버퍼 삽입 연산
        self.rear = (self.rear+1)%self.capacity   # 무조건 삽입
        self.array[self.rear] = item
        if self.isEmpty():  # front == rear
            self.front = (self.front+1)%self.capacity

- 2. 링 버퍼 테스트 프로그램

In [ ]:
import random
q = ArrayQueue(8)               # 큐 객체를 생성(capacity=8)

q.display("초기 상태")
for i in range(6) :             # enqueue2(): 0, 1, 2, 3, 4, 5
    q.enqueue2(i)
q.display("삽입 0-5")

q.enqueue2(6); q.enqueue2(7)    # enqueue2(): 6, 7
q.display("삽입 6,7")           # 포화 발생

q.enqueue2(8); q.enqueue2(9)    # enqueue2(): 8, 9
q.display("삽입 8,9")

q.dequeue(); q.dequeue()        # dequeue() 2회
q.display("삭제  x2")



---



### 3-4. **덱(Deque)**

- 덱(deque) : double-ended queue
- 전단과 후단에서 모두 삽입과 삭제가 가능한 큐

#### **덱의 연산**

| 연산 | 설명 |
| :--- | :--- |
| **addFront(e)** | 새로운 요소 e를 전단(Front)에 추가 |
| **addRear(e)** | 새로운 요소 e를 후단(Rear)에 추가 |
| **deleteFront()** | 덱의 전단 요소를 꺼내서 반환 |
| **deleteRear()** | 덱의 후단 요소를 꺼내서 반환 |
| **getFront()** | 덱의 전단 요소를 삭제하지 않고 반환 |
| **getRear()** | 덱의 후단 요소를 삭제하지 않고 반환 |
| **isEmpty()** | 덱이 비어 있는지 확인하여 True/False 반환 |
| **isFull()** | 덱이 가득 차 있는지 확인하여 True/False 반환 |
| **size()** | 덱에 들어 있는 전체 요소의 수 반환 |

- addRear, deleteFront, getFront: 원형 큐 연산을 호출
    - 원형 큐 enqueue, dequeue, peek 연산 호출
- addFront, deleteRear, getRear **새로 구현** 필요

### **[실습]: 원형 큐 상속을 이용한 덱 구현하기**

- 1. 덱 클래스 생성

In [ ]:
# 원형 큐 클래스 상속받아 원형 덱 클래스 생성
class CircularDeque(ArrayQueue) :
    def __init__( self, capacity=10 ) :
        super().__init__(capacity)

    # 원형 덱: 동작이 동일한 연산들
    def addRear( self, item ): self.enqueue(item )
    def deleteFront( self ): return self.dequeue()
    def getFront( self ): return self.peek()

    # 원형 덱: 추가된 연산들
    def addFront( self, item ):
        if not self.isFull():
            self.array[self.front] = item
            self.front = (self.front - 1 + self.capacity) % self.capacity # 전단 회전
        else: pass

    def deleteRear( self ):
        if not self.isEmpty():
            item = self.array[self.rear];
            self.rear = (self.rear - 1 + self.capacity) % self.capacity  # 후단 회전
            return item
        else: pass

    def getRear( self ):
        if not self.isEmpty():
            return self.array[self.rear]
        else: pass

- 2. 덱 테스트 프로그램

In [ ]:
dq = CircularDeque()

for i in range(9):
    if i%2==0 : dq.addRear(i)
    else : dq.addFront(i)
dq.display("홀수는 전단, 짝수는 후단 삽입")

for i in range(2): dq.deleteFront()
for i in range(3): dq.deleteRear()
dq.display("전단 삭제 2번, 후단 삭제 3번")

for i in range(9,14): dq.addFront(i)
dq.display("전단에 9 ~ 13 삽입")

#### **[참고] 파이썬에서 큐와 덱 사용하기**

##### 방법 1) queue 모듈의 Queue 사용하기

In [ ]:
import random                   # 난수 발생을 위해 random 모듈 포함
import queue                    # 파이썬의 큐 모듈 포함
q = queue.Queue(8)              # 큐 객체를 생성(capacity=8)

print("삽입 순서: ", end='')
while not q.full() :            # 큐에 빈 칸인 남았으면
    v = random.randint(0,100)   # 0~99사이의 정수 발생
    q.put(v)                    # 삽입
    print(v, end=' ')
print()

print("삭제 순서: ", end='')
while not q.empty() :         # 큐에 요소가 남아 있으면
    print(q.get(), end=' ')   # 꺼내서 화면에 출력
print()

##### 방법 2) collections모듈의 deque 클래스 사용하기

In [ ]:
import collections              # 덱을 사용하기 위해 collections 모듈 포함
dq = collections.deque()        # 덱 객체를 생성

print("덱은 공백상태 아님" if dq else "덱은 공백상태")
for i in range(9):
    if i%2==0 : dq.append(i)
    else : dq.appendleft(i)
print("홀수는 전단, 짝수는 후단 삽입", dq)

for i in range(2): dq.popleft()
for i in range(3): dq.pop()
print("전단 삭제 2번, 후단 삭제 3번", dq)

for i in range(9,14): dq.appendleft(i)
print("전단에 9 ~ 13 삽입          ", dq)

print("덱은 공백상태 아님" if dq else "덱은 공백상태")



---



### 3-5. **리스트(List)**

- 가장 자유로운 **선형 자료구조**(**연결 자료구조**)
- 각 자료는 순서 또는 위치(position)를 가짐 - 다양한 항목들을 저장, 조회할 수 있음
- **장점**은 동적 메모리 할당이 가능
- **단점**은 배열에 비해 노드 접근 시간이 느리고, 추가 메모리 공간(링크 필드)이 필요함

#### **리스트의 주요 연산**

 연산 | 설명 |
----|-----|
 **insert(pos, e)** | pos 위치에 새로운 데이터(요소) 삽입 |
 **delete(pos)** | pos 위치에 있는 요소 꺼내서 반환 |
 **getEntry(pos)** |  pos 위치에 있는 요소를 삭제하지 않고 반환 |
 **isEmpty()** | 리스트가 비어 있는지 여부 반환, True/False 반환 |
 **isFull()** |  리스트가 가득 차 있는지 확인, True/False 반환 |
 **size()** |  리스트에 들어 있는 전체 요소의 수 반환 |

#### **리스트의 연산 동작**

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data  # 데이터
        self.link = None  # 연결 링크

class LinkedList:
    def __init__(self):
        self.head = None  # 헤더 포인터

    def insert(self, pos, e): # 삽입 연산
        new_node = Node(e)                 # 신규 노드 추가
        if pos == 0:                       # 위치가 처음이면
            new_node.link = self.head      # 신규 노드 링크는 헤드 포인터(null)로 지정
            self.head = new_node           # 헤드 포인터는 신규 노드를 가르키도록 지정
            return                         # 반환
        current = self.head                # 추가 위치 찾기 위해 처음 지정하여 순회
        count = 1                          #
        while current and count < pos:     #
            current = current.link         # 다음 진행
            count += 1                     #
        if current is None:
            raise IndexError("Index out of bounds")
        new_node.link = current.link    # 현재 포인터
        current.link = new_node

    def delete(self, pos):       # 삭제 연산
        if self.head is None:
            raise IndexError("List is empty")
        if pos == 0:
            self.head = self.head.link
            return
        current = self.head                        # 삭제 위치 찾기 위해 처음 지정하여 순회
        count = 1                                  #
        while current.link and count < pos:        #
            current = current.link                 # 다음 진행
            count += 1                             #
        if current.link is None:
            raise IndexError("Index out of bounds")
        current.link = current.next.link


#### **파이썬 리스트는 배열 구조 리스트**
- 파이썬 리스트는 배열 구조 리스트
- 용량이 제한되지 않도록 **동적 배열로 구현**됨
- 용량 확장은 내부적으로 처리되므로 사용자는 신경을 쓰지 않아도 됨
- 파이썬 리스트의 append() 연산의 처리 시간은 항상 동일하지 않음


| 분류 | 연산(함수) | 시간 복잡도 | 설명 |
| :--- | :--- | :--- | :--- |
| **요소 추가** | `append(x)` | $O(1)$ | 리스트 끝에 요소 하나를 추가 (평균) |
| | `extend(k)` | $O(k)$ | 리스트 끝에 다른 리스트(k개 요소)를 연결 |
| | `insert(i, x)` | $O(n)$ | 특정 인덱스 `i`에 요소를 삽입 (이후 요소들 이동 발생) |
| **검색** | `count(x)` | $O(n)$ | 리스트 전체를 순회하며 특정 요소 `x`의 개수를 확인 |
| | `index(x)` | $O(n)$ | 특정 요소 `x`가 처음 나타나는 인덱스를 반환 |
| **요소 제거** | `pop()` | $O(1)$ | 리스트의 마지막 요소를 제거하고 반환 |
| | `pop(i)` | $O(n)$ | 특정 인덱스 `i`의 요소를 제거 (이후 요소들 앞당기기 발생) |
| | `remove(x)` | $O(n)$ | 특정 값을 가진 첫 번째 요소를 삭제 |
| **리스트 조작** | `reverse()` | $O(n)$ | 리스트의 요소 순서를 반대로 뒤집음 |
| | `sort()` | $O(n \log n)$ | 리스트 요소를 정렬 (Timsort 알고리즘 사용) |


In [ ]:
import sys

lst = []
for i in range(9):
    lst.append(i)
    print(f"길이:{len(lst)}, 실제할당:{lst.__sizeof__()//8-4}칸")